In [1]:
# --- LangChain and LLM Imports ---
from langchain_openai import ChatOpenAI 

# --- Document Loading and Vector Store ---
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from dotenv import load_dotenv
import tiktoken
import json
import os
from time import monotonic

# --- Datasets and Typing ---
from datasets import Dataset
from typing_extensions import TypedDict
from IPython.display import display, Image
from typing import List, TypedDict
from rank_bm25 import BM25Okapi
# --- Helper Functions ---
from helper_functions import (
    num_tokens_from_string,
    replace_t_with_space,
    replace_double_lines_with_one_line,
    split_into_chapters,
    analyse_metric_results,
    escape_quotes,
    text_wrap,
    extract_book_quotes_as_documents
)

# --- Load environment variables (e.g., API keys) ---
load_dotenv()

# --- Set environment variable for debugging (optional) ---
os.environ["PYDEVD_WARN_EVALUATION_TIMEOUT"] = "100000"

In [2]:
hp_pdf_path = "docs/hp/Harry Potter - Book 1 - The Sorcerers Stone.pdf"

In [34]:
# --- Split the PDF into chapters and preprocess the text ---
# 1. Split the PDF into chapters using the provided helper function.
#    This function takes the path to the PDF and returns a list of Document objects, each representing a chapter.
chapters = split_into_chapters(hp_pdf_path)
# 2. Clean up the text in each chapter by replacing unwanted characters (e.g., '\t') with spaces.
#    This ensures the text is consistent and easier to process downstream.
chapters = replace_t_with_space(chapters)
# 3. Print the number of chapters extracted to verify the result.
print(len(chapters))

17


In [35]:
chapters[0]

Document(metadata={'chapter': 1}, page_content="CHAPTER ONE\n \nTHE BOY WHO LIVED\n \n      \nM \nr. and Mrs. Dursley, of number four, Privet Drive, were proud to say\nthat they were perfectly normal, thank you very much. They were the last people\nyou’d expect to be involved in anything strange or mysterious, because they just\ndidn’t hold with such nonsense.\n      Mr. Dursley was the director of a firm called Grunnings, which made\ndrills. He was a big, beefy man with hardly any neck, although he did have a\nvery large mustache. Mrs. Dursley was thin and blonde and had nearly twice the\nusual amount of neck, which came in very useful as she spent so much of her\ntime craning over garden fences, spying on the neighbors. The Dursleys had a\nsmall son called Dudley and in their opinion there was no finer boy anywhere.\n      The Dursleys had everything they wanted, but they also had a secret, and\ntheir greatest fear was that somebody would discover it. They didn’t think they\ncould be

In [36]:
encoding = tiktoken.encoding_for_model("gpt-4o")
text = chapters[0].page_content
token_integers = encoding.encode(text)
token_count = len(token_integers)
print(f"Token Count: {token_count}")

Token Count: 6697


In [37]:
# --- Load and Preprocess the PDF, then Extract Quotes ---
# 1. Load the PDF using PyPDFLoader
loader = PyPDFLoader(hp_pdf_path)
document = loader.load()
# 2. Clean the loaded document by replacing unwanted characters (e.g., '\t') with spaces
document_cleaned = replace_t_with_space(document)
# 3. Extract a list of quotes from the cleaned document as Document objects
book_quotes_list = extract_book_quotes_as_documents(document_cleaned)
print(len(book_quotes_list))

1534


In [38]:
book_quotes_list[1000]

Document(metadata={}, page_content=' said Wood. "We\'ve just got to make sure we play a clean game, so Snape hasn\'t got an excuse to pick on us.')

In [3]:
azure_llm_key = os.getenv("azure_llm_key")
llm = ChatOpenAI(
    model="DeepSeek-V4-Flash",
    base_url="https://3t-ai-resource.services.ai.azure.com/openai/v1",
    api_key=azure_llm_key,
    max_tokens=2048,
    temperature=0.1,
)

In [11]:
summarization_prompt_template = """Write an extensive summary of the following:
{text}
SUMMARY:"""
summarization_prompt = PromptTemplate(
    template=summarization_prompt_template,
    input_variables=["text"]
)

In [12]:
def create_chapter_summary(chapter, llm):
    """
    Creates a summary of a chapter using a large language model (LLM).
    Args:
        chapter: A Document object representing the chapter to summarize.
    Returns:
        A Document object containing the summary of the chapter.
    """

    # Extract the text content from the chapter
    chapter_txt = chapter.page_content
    max_tokens = 16000  # Maximum token limit for the model
    verbose = False  # Set to True for more detailed output
    # Calculate the number of tokens in the chapter text
    num_tokens = num_tokens_from_string(chapter_txt, 'gpt-4o')

    # Choose the summarization chain type based on token count
    if num_tokens < max_tokens:
        # For shorter chapters, use the "stuff" chain type
        chain = load_summarize_chain(
            llm,
            chain_type="stuff",
            prompt=summarization_prompt,
            verbose=verbose
        )
    else:
        # For longer chapters, use the "map_reduce" chain type
        chain = load_summarize_chain(
            llm,
            chain_type="map_reduce",
            map_prompt=summarization_prompt,
            combine_prompt=summarization_prompt,
            verbose=verbose
        )
    start_time = monotonic()
    doc_chapter = Document(page_content=chapter_txt)
    summary_result = chain.invoke([doc_chapter])
    # Print chain type and execution time for reference
    print(f"Chain type: {chain.__class__.__name__}")
    print(f"Run time: {monotonic() - start_time}")
    # Clean up the summary text (remove double newlines, etc.)
    summary_text = replace_double_lines_with_one_line(summary_result["output_text"])
    # Create a Document object for the summary, preserving chapter metadata
    doc_summary = Document(page_content=summary_text, metadata=chapter.metadata)
    return doc_summary

In [14]:
# # --- Generate Summaries for Each Chapter ---
# chapter_summaries = []
# # Iterate over each chapter in the chapters list
# for chapter in chapters:
#     summary = create_chapter_summary(chapter, llm)
#     chapter_summaries.append(summary)

Chain type: StuffDocumentsChain
Run time: 12.92240441699687
Chain type: StuffDocumentsChain
Run time: 12.95165604200156
Chain type: StuffDocumentsChain
Run time: 9.6705019580113
Chain type: StuffDocumentsChain
Run time: 56.35622020800656
Chain type: StuffDocumentsChain
Run time: 7.771511041995836
Chain type: StuffDocumentsChain
Run time: 63.88727658300195
Chain type: StuffDocumentsChain
Run time: 452.6376131249999
Chain type: StuffDocumentsChain
Run time: 23.597769999993034
Chain type: StuffDocumentsChain
Run time: 10.032964792000712
Chain type: StuffDocumentsChain
Run time: 17.540286165996804
Chain type: StuffDocumentsChain
Run time: 7.838023917007376
Chain type: StuffDocumentsChain
Run time: 40.02960749999329
Chain type: StuffDocumentsChain
Run time: 31.28564929199638
Chain type: StuffDocumentsChain
Run time: 20.957695416000206
Chain type: StuffDocumentsChain
Run time: 17.759518374994514
Chain type: StuffDocumentsChain
Run time: 51.09342441699118
Chain type: StuffDocumentsChain
Run t

In [20]:
# docs_json = [doc.model_dump() for doc in chapter_summaries]
# with open("processed_docs/hp/chapter_summary.json", "w", encoding="utf-8") as f:
#     json.dump(docs_json, f, ensure_ascii=False, indent=4)

In [21]:
with open("processed_docs/hp/chapter_summary.json", "r", encoding="utf-8") as f:
    docs_json = json.load(f)
chapter_summaries = [Document(**d) for d in docs_json]

In [22]:
chapter_summaries[0]

Document(metadata={'chapter': 1}, page_content='Here is an extensive summary of Chapter One of *Harry Potter and the Sorcerer\'s Stone*.\n### Chapter One: The Boy Who Lived\nThe chapter opens by introducing the Dursley family of number four, Privet Drive. Mr. and Mrs. Dursley are a profoundly ordinary, conventional, and proud couple who despise anything strange or mysterious. Mr. Dursley is a large, beefy man who works as a director at a drill-making company, while his wife, Petunia, is thin, blonde, and spends her time spying on the neighbors. They have a spoiled son, Dudley, whom they believe to be the perfect child. The Dursleys harbor a deep, shameful secret: Petunia’s sister, Lily, and her husband, James Potter, are the very opposite of "Dursleyish." The Dursleys have cut off all contact with them, fearing what the neighbors would think of these "unDursleyish" relatives and their child, Harry.\nThe story begins on a dull, gray Tuesday. As Mr. Dursley leaves for work, he notices th

In [3]:
embedding_base_url = os.getenv("embedding_base_url")
embedding_key = os.getenv("embedding_key")
embedding_deployment = os.getenv("embedding_deployment")
embeddings = OpenAIEmbeddings(
    model=embedding_deployment,
    base_url=f"{embedding_base_url}/openai/v1",
    api_key=embedding_key,
)

In [27]:
def encode_book(path, embeddings, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a FAISS vector store using OpenAI embeddings.
    Args:
        path (str): The path to the PDF file.
        chunk_size (int): The desired size of each text chunk.
        chunk_overlap (int): The amount of overlap between consecutive chunks.
    Returns:
        FAISS: A FAISS vector store containing the encoded book content.
    """
    # 1. Load the PDF document using PyPDFLoader
    loader = PyPDFLoader(path)
    documents = loader.load()

    # 2. Split the document into chunks for embedding
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    texts = text_splitter.split_documents(documents)
    # 3. Clean up the text chunks (replace unwanted characters)
    cleaned_texts = replace_t_with_space(texts)
    # embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)
    return vectorstore

In [28]:
def encode_chapter_summaries(chapter_summaries, embeddings):
    """
    Encodes a list of chapter summaries into a FAISS vector store using OpenAI embeddings.
    Args:
        chapter_summaries (list): A list of Document objects representing the chapter summaries.
    Returns:
        FAISS: A FAISS vector store containing the encoded chapter summaries.
    """
    # Encode the chapter summaries into a FAISS vector store
    chapter_summaries_vectorstore = FAISS.from_documents(chapter_summaries, embeddings)
    # Return the vector store
    return chapter_summaries_vectorstore

In [29]:
def encode_quotes(book_quotes_list, embeddings):
    """
    Encodes a list of book quotes into a FAISS vector store using OpenAI embeddings.
    Args:
        book_quotes_list (list): A list of Document objects, each representing a quote from the book.
    Returns:
        FAISS: A FAISS vector store containing the encoded book quotes.
    """
    # Encode the book quotes into a FAISS vector store
    quotes_vectorstore = FAISS.from_documents(book_quotes_list, embeddings)
    return quotes_vectorstore

In [31]:
# --- Create or Load Vector Stores for Book Chunks, Chapter Summaries, and Book Quotes ---
# Check if the vector stores already exist on disk
if (
    os.path.exists("embedding/chunks_vector_store") and
    os.path.exists("embedding/chapter_summaries_vector_store") and
    os.path.exists("embedding/book_quotes_vectorstore")
):
    # If vector stores exist, load them using OpenAI embeddings
    chunks_vector_store = FAISS.load_local(
        "embedding/chunks_vector_store", embeddings, allow_dangerous_deserialization=True
    )
    chapter_summaries_vector_store = FAISS.load_local(
        "embedding/chapter_summaries_vector_store", embeddings, allow_dangerous_deserialization=True
    )
    book_quotes_vectorstore = FAISS.load_local(
        "embedding/book_quotes_vectorstore", embeddings, allow_dangerous_deserialization=True
    )
else:
    # If vector stores do not exist, encode and save them
    # 1. Encode the book into a vector store of chunks
    chunks_vector_store = encode_book(hp_pdf_path, embeddings, chunk_size=1000, chunk_overlap=200)

    # 2. Encode the chapter summaries into a vector store
    chapter_summaries_vector_store = encode_chapter_summaries(chapter_summaries, embeddings)

    # 3. Encode the book quotes into a vector store
    book_quotes_vectorstore = encode_quotes(book_quotes_list, embeddings)

    # 4. Save the vector stores to disk for future use
    chunks_vector_store.save_local("embedding/chunks_vector_store")
    chapter_summaries_vector_store.save_local("embedding/chapter_summaries_vector_store")
    book_quotes_vectorstore.save_local("embedding/book_quotes_vectorstore")

In [39]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)
texts = text_splitter.split_documents(document_cleaned)

In [40]:
cleaned_texts = replace_t_with_space(texts)

In [41]:
import re
for doc in cleaned_texts:
    doc.page_content = re.sub(r'\n', ' ', doc.page_content)

In [42]:
cleaned_texts[20]

Document(metadata={'producer': 'calibre 3.42.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.42.0 [https://calibre-ebook.com]', 'creationdate': '2019-07-26T17:37:04+00:00', 'author': 'J.K.Rowling', 'title': "Harry Potter 1 - Harry Potter and the Sorcerer's Stone", 'source': 'docs/hp/Harry Potter - Book 1 - The Sorcerers Stone.pdf', 'total_pages': 221, 'page': 9, 'page_label': '10'}, page_content='nearest street lamp went out with a little pop. He clicked it again — the next lamp flickered into darkness. Twelve times he clicked the Put-Outer, until the only lights left on the whole street were two tiny pinpricks in the distance, which were the eyes of the cat watching him. If anyone looked out of their window now, even beady-eyed Mrs. Dursley, they wouldn’t be able to see anything that was happening down on the pavement. Dumbledore slipped the Put-Outer back inside his cloak and set off down the street toward number four, where he sat down on the wall next to the cat. He didn’t lo

In [45]:
doc_content = [doc.page_content for doc in cleaned_texts]
tokenized_docs = [doc.lower().split() for doc in doc_content]
bm25 = BM25Okapi(tokenized_docs)

In [47]:
tokenized_docs[20]

['nearest',
 'street',
 'lamp',
 'went',
 'out',
 'with',
 'a',
 'little',
 'pop.',
 'he',
 'clicked',
 'it',
 'again',
 '—',
 'the',
 'next',
 'lamp',
 'flickered',
 'into',
 'darkness.',
 'twelve',
 'times',
 'he',
 'clicked',
 'the',
 'put-outer,',
 'until',
 'the',
 'only',
 'lights',
 'left',
 'on',
 'the',
 'whole',
 'street',
 'were',
 'two',
 'tiny',
 'pinpricks',
 'in',
 'the',
 'distance,',
 'which',
 'were',
 'the',
 'eyes',
 'of',
 'the',
 'cat',
 'watching',
 'him.',
 'if',
 'anyone',
 'looked',
 'out',
 'of',
 'their',
 'window',
 'now,',
 'even',
 'beady-eyed',
 'mrs.',
 'dursley,',
 'they',
 'wouldn’t',
 'be',
 'able',
 'to',
 'see',
 'anything',
 'that',
 'was',
 'happening',
 'down',
 'on',
 'the',
 'pavement.',
 'dumbledore',
 'slipped',
 'the',
 'put-outer',
 'back',
 'inside',
 'his',
 'cloak',
 'and',
 'set',
 'off',
 'down',
 'the',
 'street',
 'toward',
 'number',
 'four,',
 'where',
 'he',
 'sat',
 'down',
 'on',
 'the',
 'wall',
 'next',
 'to',
 'the',
 'cat.'

In [53]:
import numpy as np
query = "who is harry potter ?"
tokenized_query = query.lower().split()
BM25_K = 50
bm25_ranked_indices = np.argpartition(
    bm25_scores,
    -BM25_K
)[-BM25_K:]
bm25_ranked_indices = bm25_ranked_indices[
    np.argsort(bm25_scores[bm25_ranked_indices])[::-1]
]
bm25_ranked_indices

array([  0, 517,  36,  65,   1, 326,  37, 361, 126, 261, 388, 576, 433,
       177, 459, 227,  97, 486, 525, 286,  28, 508,  27, 598, 279, 372,
         9, 218,  88, 519,  64, 516, 521, 195, 596, 405, 584, 254, 520,
       252, 524, 522, 440, 531, 198, 276, 295, 199, 112,  39])

In [58]:
from collections import defaultdict
rrf_scores = defaultdict(float)
for rank, idx in enumerate(bm25_ranked_indices):
    rrf_scores[int(idx)] += 1/(rrf_k + rank + 1)

In [59]:
rrf_scores

defaultdict(float,
            {0: 0.01639344262295082,
             517: 0.016129032258064516,
             36: 0.015873015873015872,
             65: 0.015625,
             1: 0.015384615384615385,
             326: 0.015151515151515152,
             37: 0.014925373134328358,
             361: 0.014705882352941176,
             126: 0.014492753623188406,
             261: 0.014285714285714285,
             388: 0.014084507042253521,
             576: 0.013888888888888888,
             433: 0.0136986301369863,
             177: 0.013513513513513514,
             459: 0.013333333333333334,
             227: 0.013157894736842105,
             97: 0.012987012987012988,
             486: 0.01282051282051282,
             525: 0.012658227848101266,
             286: 0.0125,
             28: 0.012345679012345678,
             508: 0.012195121951219513,
             27: 0.012048192771084338,
             598: 0.011904761904761904,
             279: 0.011764705882352941,
             372: 0.0

In [1]:
from doc_processing import process_pdf_file
doc_path = "docs/hp/Harry Potter - Book 1 - The Sorcerers Stone.pdf"
document_cleaned, book_quotes_list, chapters = process_pdf_file(doc_path, True, True)

In [2]:
len(document_cleaned)

221

In [4]:
document_cleaned[10]

Document(metadata={'producer': 'calibre 3.42.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.42.0 [https://calibre-ebook.com]', 'creationdate': '2019-07-26T17:37:04+00:00', 'author': 'J.K.Rowling', 'title': "Harry Potter 1 - Harry Potter and the Sorcerer's Stone", 'source': 'docs/hp/Harry Potter - Book 1 - The Sorcerers Stone.pdf', 'total_pages': 221, 'page': 10, 'page_label': '11'}, page_content='“It certainly seems so,” said Dumbledore. “We have much to be thankful\nfor. Would you care for a lemon drop?”\n      “A what?”\n      “A lemon drop. They’re a kind of Muggle sweet I’m rather fond of.”\n      “No, thank you,” said Professor McGonagall coldly, as though she didn’t\nthink this was the moment for lemon drops. “As I say, even if You-Know-Who\nhas gone —”\n      “My dear Professor, surely a sensible person like yourself can call him by\nhis name? All this ‘You-Know-Who’ nonsense — for eleven years I have been\ntrying to persuade people to call him by his proper name: Voldemo

In [5]:
len(book_quotes_list)

1534

In [6]:
book_quotes_list[10]

Document(metadata={}, page_content='Potter." He decided he didn\'t dare. Instead he said, as casually as he could, ')

In [7]:
len(chapters)

17

In [8]:
chapters[10]

Document(metadata={'chapter': 11}, page_content="CHAPTER ELEVEN\n \nQUIDDITCH\n \nA \ns they entered November, the weather turned very cold. The mountains\naround the school became icy gray and the lake like chilled steel. Every morning\nthe ground was covered in frost. Hagrid could be seen from the upstairs windows\ndefrosting broomsticks on the Quidditch field, bundled up in a long moleskin\novercoat, rabbit fur gloves, and enormous beaverskin boots.\n      The Quidditch season had begun. On Saturday, Harry would be playing\nin  his  first  match  after  weeks  of  training:  Gryffindor  versus  Slytherin.  If\nGryffindor  won,  they  would  move  up  into  second  place  in  the  house\nchampionship.\n      Hardly anyone had seen Harry play because Wood had decided that, as\ntheir secret weapon, Harry should be kept, well, secret. But the news that he was\nplaying Seeker had leaked out somehow, and Harry didn’t know which was\nworse — people telling him he’d be brilliant or people t

In [4]:
chunks_vector_store = FAISS.load_local(
    "embedding/chunks_vector_store",
    embeddings,
    allow_dangerous_deserialization=True,
)
faiss_retriever = chunks_vector_store.as_retriever(search_kwargs={"k": 5})

def retrieve_chunks_context_per_question(state):
    """
    Retrieves relevant context for a given question from the book chunks.

    Args:
        state (dict): A dictionary containing the question to answer, with key "question".

    Returns:
        dict: A dictionary with keys:
            - "context": Aggregated context string from relevant book chunks.
            - "question": The original question.
    """
    print("Retrieving relevant chunks...")
    question = state["question"]
    # Retrieve relevant book chunks using the retriever
    docs = chunks_query_retriever.invoke(question)
    # Concatenate the content of the retrieved documents
    context = " ".join(doc.page_content for doc in docs)
    context = escape_quotes(context)
    return {"context": context, "question": question}

In [17]:
import pickle
def tokenize(text: str):
    return re.findall(r"\b\w+\b", text.lower())
with open("embedding/bm25.pkl", "rb") as f:
    bm25_retriever = pickle.load(f)
bm25_retriever.k = 5

In [32]:
from collections import defaultdict
from langchain_core.documents import Document

def hybrid_search(
    query: str,
    bm25_retriever,
    vector_retriever,
    top_k: int = 5,
    rrf_k: int = 60,
):
    """
    Hybrid search using BM25 + Vector Search + Reciprocal Rank Fusion (RRF).

    Returns:
        top_docs      : Final fused documents
        bm25_docs     : BM25 retrieval results
        vector_docs   : Vector retrieval results
        ranked_scores : (page_content, score) tuples after RRF
    """

    # Retrieve documents
    bm25_docs = bm25_retriever.invoke(query)
    vector_docs = vector_retriever.invoke(query)

    scores = defaultdict(float)
    unique_docs = {}

    # BM25 ranking
    for rank, doc in enumerate(bm25_docs):
        key = doc.page_content

        unique_docs[key] = doc
        scores[key] += 1.0 / (rrf_k + rank + 1)

    # Vector ranking
    for rank, doc in enumerate(vector_docs):
        key = doc.page_content

        unique_docs[key] = doc
        scores[key] += 1.0 / (rrf_k + rank + 1)

    # Sort by RRF score
    ranked_scores = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Return final documents
    top_docs = [
        unique_docs[key]
        for key, _ in ranked_scores[:top_k]
    ]

    return top_docs

In [33]:
import re
query = "Who is Harrpy Potter ?"
results = hybrid_search(query, bm25_retriever, faiss_retriever)
results

[Document(metadata={'producer': 'calibre 3.42.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.42.0 [https://calibre-ebook.com]', 'creationdate': '2019-07-26T17:37:04+00:00', 'author': 'J.K.Rowling', 'title': "Harry Potter 1 - Harry Potter and the Sorcerer's Stone", 'source': 'docs/hp/Harry Potter - Book 1 - The Sorcerers Stone.pdf', 'total_pages': 221, 'page': 70, 'page_label': '71'}, page_content='The oldest boy came striding into sight. He had already changed into his\nbillowing black Hogwarts robes, and Harry noticed a shiny silver badge on his\nchest with the letter P on it.\n      “Can’t stay long, Mother,” he said. “I’m up front, the prefects have got\ntwo compartments to themselves —”\n      “Oh, are you a prefect, Percy?” said one of the twins, with an air of great\nsurprise. “You should have said something, we had no idea.”\n      “Hang on, I think I remember him saying something about it,” said the\nother twin. “Once —”\n      “Or twice —”\n      “A minute —”\n      “Al

In [35]:
vector_docs[1].page_content

'“ — yes, their son, Harry —”\n      Mr. Dursley stopped dead. Fear flooded him. He looked back at the\nwhisperers as if he wanted to say something to them, but thought better of it.\n      He dashed back across the road, hurried up to his office, snapped at his\nsecretary not to disturb him, seized his telephone, and had almost finished\ndialing his home number when he changed his mind. He put the receiver back\ndown  and  stroked  his  mustache,  thinking…no,  he  was  being  stupid.  Potter\nwasn’t such an unusual name. He was sure there were lots of people called Potter\nwho had a son called Harry. Come to think of it, he wasn’t even sure his nephew\nwas called Harry. He’d never even seen the boy. It might have been Harvey. Or\nHarold. There was no point in worrying Mrs. Dursley; she always got so upset at\nany mention of her sister. He didn’t blame her — if he’d had a sister like that…\nbut all the same, those people in cloaks.…'